# Full Chain: From PDF to Precision Search

Build a compliance search tool for the EU AI Act using Jina AI models on Elastic Inference Service.

**What you'll do:**
1. Fetch the EU AI Act with Jina Reader
2. Index it with `semantic_text` (auto-embedding via EIS)
3. Search semantically
4. Rerank for legal precision

**Time:** ~15–20 minutes

## 1. Setup & Connect

In [ ]:
!pip install elasticsearch requests -q

In [ ]:
import os
from elasticsearch import Elasticsearch

# Load from environment or prompt
ELASTICSEARCH_URL = os.environ.get("ELASTICSEARCH_URL") or input("Elasticsearch URL: ")
ELASTIC_API_KEY = os.environ.get("ELASTIC_API_KEY") or input("Elastic API Key: ")
JINA_API_KEY = os.environ.get("JINA_API_KEY") or input("Jina API Key: ")

es = Elasticsearch(ELASTICSEARCH_URL, api_key=ELASTIC_API_KEY)
print(f"Connected to Elasticsearch {es.info()['version']['number']}")

## 2. Ingest with Jina Reader

The EU AI Act is published on EUR-Lex. Jina Reader converts it to clean markdown — no scripts, no nav, no cookie banners.

In [ ]:
import requests

url = "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689"

response = requests.post(
    "https://r.jina.ai/",
    headers={
        "Authorization": f"Bearer {JINA_API_KEY}",
        "X-Return-Format": "markdown",
        "Accept": "application/json",
    },
    json={"url": url},
)
data = response.json()
markdown_text = data["data"]["content"]
print(f"Got {len(markdown_text):,} characters of clean markdown")
print(markdown_text[:500])

### Parse into articles

Split on `Article N` boundaries — each article becomes one searchable document.

In [ ]:
import re

articles = []
pattern = r"(?:^|\n)(?:#{1,4}\s*)?(?:\*{0,2})Article\s+(\d+)\b"
splits = re.split(pattern, markdown_text)

# splits alternates: [preamble, "1", text1, "2", text2, ...]
for i in range(1, len(splits) - 1, 2):
    article_num = int(splits[i])
    text = splits[i + 1].strip()
    lines = text.split("\n", 1)
    title = lines[0].strip().strip("*#. ")
    articles.append({
        "id": f"eu-ai-act-en-art-{article_num}",
        "article_number": article_num,
        "title": f"Article {article_num} — {title}",
        "text": text,
        "language": "en",
        "url": url,
    })

print(f"Parsed {len(articles)} articles")

In [ ]:
# Preview one article
art = articles[5]
print(f"\N{PAGE FACING UP} {art['title']}\n")
print(art["text"][:400] + "...")

## 3. Index with `semantic_text`

One field type handles embedding automatically. **v5-text-small is the default** — you don't even need to specify `inference_id`.

In [ ]:
INDEX = "search-eu-ai-act-demo"

# Delete if exists (for re-runs)
if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)

es.indices.create(
    index=INDEX,
    mappings={
        "properties": {
            "text": {"type": "semantic_text"},  # v5-text-small is the default!
            "title": {"type": "text"},
            "article_number": {"type": "integer"},
            "language": {"type": "keyword"},
            "url": {"type": "keyword"},
        }
    },
)
print(f"Index '{INDEX}' created with semantic_text")

In [ ]:
from elasticsearch.helpers import bulk

actions = [
    {"_index": INDEX, "_id": doc["id"], "_source": doc}
    for doc in articles
]
success, errors = bulk(es, actions, raise_on_error=False)
print(f"Indexed {success} articles ({len(errors)} errors)")

In [ ]:
# Wait for embeddings to generate, then verify
import time
time.sleep(5)

count = es.count(index=INDEX)["count"]
print(f"{count} documents in index")

## 4. Semantic Search

A normal `match` query on a `semantic_text` field automatically does semantic search.

In [ ]:
query = "Can law enforcement use facial recognition in public spaces?"

results = es.search(
    index=INDEX,
    query={"match": {"text": query}},
    size=5,
    _source=["title", "article_number"],
)

print(f"Query: {query}\n")
for i, hit in enumerate(results["hits"]["hits"]):
    print(f"  #{i+1} [{hit['_score']:.2f}] {hit['_source']['title']}")

## 5. Rerank for Precision

Semantic search found the right neighborhood. The reranker reads each document alongside the query and re-scores them.

**Key facts:**
- The reranker reads the **raw text**, not the embeddings
- It works even if your first stage was keyword search (no embeddings needed)
- The embedding endpoint is pre-configured. The reranker needs explicit creation:

In [ ]:
# Create the reranker endpoint — this is the actual API call
try:
    es.inference.put(
        inference_id="jina-reranker-v3-demo",
        task_type="rerank",
        body={
            "service": "jina",
            "service_settings": {
                "model_id": "jina-reranker-v3",
                "api_key": JINA_API_KEY,
            },
        },
    )
    print("Reranker endpoint created")
except Exception as e:
    if "already exists" in str(e).lower() or "resource_already_exists" in str(e).lower():
        print("Reranker endpoint already exists")
    else:
        raise

In [ ]:
# Same query, now with reranking
reranked = es.search(
    index=INDEX,
    retriever={
        "text_similarity_reranker": {
            "retriever": {
                "standard": {
                    "query": {"match": {"text": query}}
                }
            },
            "inference_id": "jina-reranker-v3-demo",
            "inference_text": query,
            "field": "text",
            "rank_window_size": 50,
        }
    },
    size=5,
    _source=["title", "article_number"],
)

print(f"Query: {query}\n")
print("After reranking:")
for i, hit in enumerate(reranked["hits"]["hits"]):
    print(f"  #{i+1} [{hit['_score']:.2f}] {hit['_source']['title']}")

### Side-by-side comparison

In [ ]:
naive_titles = [h["_source"]["title"] for h in results["hits"]["hits"]]
reranked_titles = [h["_source"]["title"] for h in reranked["hits"]["hits"]]

print("Semantic Search Only  vs  After Reranking\n")
for i in range(5):
    n = naive_titles[i] if i < len(naive_titles) else "—"
    r = reranked_titles[i] if i < len(reranked_titles) else "—"
    new = " NEW" if r not in naive_titles else ""
    print(f"  #{i+1}  {n}")
    print(f"   ->  {r}{new}")
    print()

Notice how the reranker surfaces the **specific exception** that allows law enforcement to use facial recognition under narrow conditions. A lawyer needs that — not just "it's banned."

## 6. Try It Yourself

Experiment with your own queries. Change the query, adjust `rank_window_size`, try with and without reranking.

In [ ]:
# Your query here:
my_query = "What are the transparency requirements for AI chatbots?"

results = es.search(index=INDEX, query={"match": {"text": my_query}}, size=5, _source=["title"])
for i, hit in enumerate(results["hits"]["hits"]):
    print(f"  #{i+1} [{hit['_score']:.2f}] {hit['_source']['title']}")

In [ ]:
# More ideas:
# "When is a conformity assessment required?"
# "What penalties exist for non-compliance?"
# "Who is responsible when an AI system causes harm?"
# "Ausnahmen fur Strafverfolgungsbehorden"  (German: law enforcement exceptions — tests multilingual!)